In [1]:
import pandas as pd
from sqlalchemy import create_engine

engine = create_engine("postgresql+psycopg2://postgres:Zxcvbnm,./123@localhost:5432/marketing")

df = pd.read_csv(r"C:\Users\HP\Desktop\Marketing.csv")  

df.to_sql("customer_data", engine, if_exists="replace", index=False)

print("Data loaded successfully!")
print(df.shape)
print(df.columns.tolist())

Data loaded successfully!
(2241, 29)
['ID', 'Year_Birth', 'Education', 'Marital_Status', 'Income', 'Kidhome', 'Teenhome', 'Dt_Customer', 'Recency', 'MntWines', 'MntFruits', 'MntMeatProducts', 'MntFishProducts', 'MntSweetProducts', 'MntGoldProds', 'NumDealsPurchases', 'NumWebPurchases', 'NumCatalogPurchases', 'NumStorePurchases', 'NumWebVisitsMonth', 'AcceptedCmp3', 'AcceptedCmp4', 'AcceptedCmp5', 'AcceptedCmp1', 'AcceptedCmp2', 'Complain', 'Z_CostContact', 'Z_Revenue', 'Response']


In [2]:

from sqlalchemy import text

query = """
ALTER TABLE customer_data RENAME COLUMN "ID" TO id;
ALTER TABLE customer_data RENAME COLUMN "Year_Birth" TO year_birth;
ALTER TABLE customer_data RENAME COLUMN "Education" TO education;
ALTER TABLE customer_data RENAME COLUMN "Marital_Status" TO marital_status;
ALTER TABLE customer_data RENAME COLUMN "Income" TO income;
ALTER TABLE customer_data RENAME COLUMN "Kidhome" TO kidhome;
ALTER TABLE customer_data RENAME COLUMN "Teenhome" TO teenhome;
ALTER TABLE customer_data RENAME COLUMN "Dt_Customer" TO dt_customer;
ALTER TABLE customer_data RENAME COLUMN "Recency" TO recency;
ALTER TABLE customer_data RENAME COLUMN "MntWines" TO mntwines;
ALTER TABLE customer_data RENAME COLUMN "MntFruits" TO mntfruits;
ALTER TABLE customer_data RENAME COLUMN "MntMeatProducts" TO mntmeatproducts;
ALTER TABLE customer_data RENAME COLUMN "MntFishProducts" TO mntfishproducts;
ALTER TABLE customer_data RENAME COLUMN "MntSweetProducts" TO mntsweetproducts;
ALTER TABLE customer_data RENAME COLUMN "MntGoldProds" TO mntgoldprods;
ALTER TABLE customer_data RENAME COLUMN "NumDealsPurchases" TO numdealspurchases;
ALTER TABLE customer_data RENAME COLUMN "NumWebPurchases" TO numwebpurchases;
ALTER TABLE customer_data RENAME COLUMN "NumCatalogPurchases" TO numcatalogpurchases;
ALTER TABLE customer_data RENAME COLUMN "NumStorePurchases" TO numstorepurchases;
ALTER TABLE customer_data RENAME COLUMN "NumWebVisitsMonth" TO numwebvisitsmonth;
ALTER TABLE customer_data RENAME COLUMN "AcceptedCmp3" TO acceptedcmp3;
ALTER TABLE customer_data RENAME COLUMN "AcceptedCmp4" TO acceptedcmp4;
ALTER TABLE customer_data RENAME COLUMN "AcceptedCmp5" TO acceptedcmp5;
ALTER TABLE customer_data RENAME COLUMN "AcceptedCmp1" TO acceptedcmp1;
ALTER TABLE customer_data RENAME COLUMN "AcceptedCmp2" TO acceptedcmp2;
ALTER TABLE customer_data RENAME COLUMN "Complain" TO complain;
ALTER TABLE customer_data RENAME COLUMN "Z_CostContact" TO z_costcontact;
ALTER TABLE customer_data RENAME COLUMN "Z_Revenue" TO z_revenue;
ALTER TABLE customer_data RENAME COLUMN "Response" TO response;
"""

# Execute SQL commands
with engine.begin() as connection:
    for statement in query.strip().split(";"):
        if statement.strip():
            connection.execute(text(statement))

print("Columns renamed successfully!")

Columns renamed successfully!


In [3]:
# Overall statistics of the dataset

query1 = """
	SELECT COUNT(*) AS Total_Customers,
	ROUND(AVG(Income)::numeric, 2) AS Average_Income,
	ROUND(AVG(Recency)::numeric, 2) AS Average_Recency,
	SUM(response) AS Total_Response_of_Last_Campaign
	FROM customer_data;
    """
Overall_stats = pd.read_sql(query1, engine)
print(Overall_stats)


   total_customers  average_income  average_recency  \
0             2241         52223.7            49.09   

   total_response_of_last_campaign  
0                            334.0  


In [4]:
# Education-wise statistics of the dataset
query2 = """
SELECT education, COUNT(*) AS Total_Customers,
	ROUND(AVG(Income)::numeric, 2) AS Average_Income,
	ROUND(AVG(Recency)::numeric, 2) AS Average_Recency,
	SUM(response) AS Total_Response_of_Last_Campaign
FROM customer_data
GROUP BY education;
"""
Education_stats = pd.read_sql(query2, engine)
print(Education_stats)

    education  total_customers  average_income  average_recency  \
0  Graduation             1127        52720.37            50.04   
1       Basic               54        20306.26            48.44   
2         PhD              486        56145.31            48.48   
3    2n Cycle              203        47633.19            48.42   
4           0                1           24.00             0.00   
5      Master              370        52917.53            47.59   

   total_response_of_last_campaign  
0                            152.0  
1                              2.0  
2                            101.0  
3                             22.0  
4                              0.0  
5                             57.0  


In [5]:
# Marital Status-wise statistics of the dataset
query3 = """
SELECT marital_status, COUNT(*) AS Total_Customers,
	ROUND(AVG(Income)::numeric, 2) AS Average_Income,
	ROUND(AVG(Recency)::numeric, 2) AS Average_Recency,
	SUM(response) AS Total_Response_of_Last_Campaign
FROM customer_data
GROUP BY marital_status;
"""
Marital_status = pd.read_sql(query3, engine)
print(Marital_status)

  marital_status  total_customers  average_income  average_recency  \
0         Single              480        50995.35            49.51   
1         Absurd                2        72365.50            53.00   
2              0                1           24.00             0.00   
3        Married              864        51724.98            48.28   
4          Alone                3        43789.00            30.33   
5       Divorced              232        52834.23            49.49   
6          Widow               77        56481.55            49.14   
7           YOLO                2        48432.00             3.00   
8       Together              580        53245.53            50.11   

   total_response_of_last_campaign  
0                            106.0  
1                              1.0  
2                              0.0  
3                             98.0  
4                              1.0  
5                             48.0  
6                             19.0  
7  

In [6]:
# Age-wise statistics of the dataset
query4 = """
SELECT CASE 
    WHEN (2014 - Year_Birth) BETWEEN 18 AND 30 THEN '18-30'
    WHEN (2014 - Year_Birth) BETWEEN 31 AND 45 THEN '31-45'
    WHEN (2014 - Year_Birth) BETWEEN 46 AND 60 THEN '46-60'
    ELSE '60+'
END AS age_group,
COUNT(*) AS Total_Customers,
	ROUND(AVG(Income)::numeric, 2) AS Average_Income,
	ROUND(AVG(Recency)::numeric, 2) AS Average_Recency,
	SUM(response) AS Total_Response_of_Last_Campaign
FROM customer_data
GROUP BY age_group
ORDER BY age_group;
"""
Age_stats = pd.read_sql(query4, engine)
print(Age_stats)

  age_group  total_customers  average_income  average_recency  \
0     18-30              259        45236.01            49.07   
1     31-45              976        49964.51            47.94   
2     46-60              736        55270.11            50.27   
3       60+              270        58795.37            50.01   

   total_response_of_last_campaign  
0                             47.0  
1                            149.0  
2                             95.0  
3                             43.0  


In [7]:
# Income-wise statistics of the dataset
query5 = """
SELECT CASE 
	WHEN Income <10000 THEN '<10k'
	WHEN Income BETWEEN 11000 AND 20000 THEN '11k - 20k'
	WHEN Income BETWEEN 21000 AND 30000 THEN '21k - 30k'
	WHEN Income BETWEEN 31000 AND 40000 THEN '31k - 40k'
	WHEN Income BETWEEN 41000 AND 50000 THEN '41k - 50k'
	ELSE '>50k'
END AS Income_Range,
COUNT(*) AS Total_Customers,
	ROUND(AVG(Income)::numeric, 2) AS Average_Income,
	ROUND(AVG(Recency)::numeric, 2) AS Average_Recency,
	SUM(response) AS Total_Response_of_Last_Campaign
FROM customer_data
GROUP BY Income_Range
ORDER BY Income_Range;
"""
Income_stats = pd.read_sql(query5, engine)
print(Income_stats)

  income_range  total_customers  average_income  average_recency  \
0         <10k               30         6560.67            44.30   
1         >50k             1273        66702.22            49.08   
2    11k - 20k               94        16848.47            50.97   
3    21k - 30k              227        25535.22            50.86   
4    31k - 40k              320        35677.04            47.93   
5    41k - 50k              297        45370.84            48.89   

   total_response_of_last_campaign  
0                              3.0  
1                            216.0  
2                             10.0  
3                             25.0  
4                             43.0  
5                             37.0  


In [8]:
# Spending Behavior Analysis
# Average Spending per Customer + Total Spending
query6 = """
SELECT SUM(MntWines + MntFruits + MntMeatProducts + MntFishProducts + MntSweetProducts + MntGoldProds) AS Overall_Total_spending,
		ROUND(AVG(MntWines + MntFruits + MntMeatProducts + MntFishProducts + MntSweetProducts + MntGoldProds), 2) AS Average_total_spending
FROM customer_data;
"""
Avg_and_total = pd.read_sql(query6, engine)
print(Avg_and_total)

# Overall Spending Behavior Analysis
query7 = """
SELECT SUM (MntWines) AS Total_Wines,
		SUM (MntFruits) AS Ttoal_Fruits,
		SUM (MntMeatProducts) AS Total_Meat_Products,
		SUM (MntFishProducts) AS Total_Fish_Products,
		SUM (MntSweetProducts) AS Total_Sweet_Products,
		SUM (MntGoldProds) AS Total_Gold
FROM customer_data;
"""
Overall_spending = pd.read_sql(query7, engine)
print(Overall_spending) 


   overall_total_spending  average_total_spending
0               1356988.0                  605.53
   total_wines  ttoal_fruits  total_meat_products  total_fish_products  \
0     680816.0       58917.0             373968.0              84057.0   

   total_sweet_products  total_gold  
0               60621.0     98609.0  


In [9]:
# Campaign Performance Analysis
query8 = """
SELECT ROUND(SUM(acceptedcmp1) * 100.0 / COUNT(*),2) AS cmp1_rate,
		ROUND(SUM(acceptedcmp2) * 100.0 / COUNT(*),2) AS cmp2_rate,
		ROUND(SUM(acceptedcmp3) * 100.0 / COUNT(*),2) AS cmp3_rate,
		ROUND(SUM(acceptedcmp4) * 100.0 / COUNT(*),2) AS cmp4_rate,
		ROUND(SUM(acceptedcmp5) * 100.0 / COUNT(*),2) AS cmp5_rate,
		ROUND(SUM(response) * 100.0 / COUNT(*),2) AS response_rate
FROM customer_data;
"""
Campaign_performance = pd.read_sql(query8, engine)
print(Campaign_performance)

   cmp1_rate  cmp2_rate  cmp3_rate  cmp4_rate  cmp5_rate  response_rate
0       6.43       1.34       7.27       7.45       7.27           14.9


In [10]:
# Purchse Channel Analysis
query9 = """
SELECT ROUND(AVG(numwebpurchases),2) AS avg_web_purchase,
ROUND(AVG(numcatalogpurchases),2) AS avg_catalog_purchase,
ROUND(AVG(numstorepurchases),2) AS avg_stote_purchase,
ROUND(AVG(numwebvisitsmonth),2) AS avg_web_visit
FROM customer_data;
"""
Purchase_channel = pd.read_sql(query9, engine)
print(Purchase_channel)


   avg_web_purchase  avg_catalog_purchase  avg_stote_purchase  avg_web_visit
0              4.08                  2.66                5.79           5.31
